# Analyze AIT Log v2 Attack Phases

In [2]:
import pandas as pd

from utils import analyze_phase

## Load MSA Data

In [3]:
dataset = "aitv2"
scenario = "fox"
data_dir = f"../data/interim/{dataset}/{scenario}/flows_labeled"
file_path = f"{data_dir}/all_flows_labeled.csv"

In [4]:
df = pd.read_csv(file_path)

In [5]:
df.head()

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase
0,f1921,1.642205e+09,1.642205e+09,0.099298,10.35.35.206,60538,91.189.91.157,123,udp,ntp,...,76,1,76,-,17,NTP,intranet_server,2022-01-15 00:00:00.869369984,2022-01-15 00:00:00.968667984,0
1,f14196,1.642205e+09,1.642205e+09,0.098276,192.168.128.4,60538,91.189.91.157,123,udp,ntp,...,76,1,76,-,17,NTP,inet_firewall,2022-01-15 00:00:00.870096922,2022-01-15 00:00:00.968372822,0
2,f3334,1.642205e+09,1.642205e+09,0.053324,172.17.129.140,43702,10.35.33.111,445,tcp,gssapi,...,514,4,379,-,6,benign_share,cloud_share,2022-01-15 00:00:02.130912066,2022-01-15 00:00:02.184236050,0
3,f5126,1.642205e+09,1.642205e+09,0.051889,172.17.129.140,43702,10.35.33.111,445,tcp,gssapi,...,514,4,379,-,6,benign_share,internal_share,2022-01-15 00:00:02.131725073,2022-01-15 00:00:02.183614016,0
4,f3335,1.642205e+09,1.642205e+09,0.012432,172.17.129.140,43704,10.35.33.111,445,tcp,gssapi,...,514,4,379,-,6,benign_share,cloud_share,2022-01-15 00:00:02.226455927,2022-01-15 00:00:02.238888025,0


In [6]:
df.value_counts("phase")

phase
0    400024
2     56443
1     14678
4        61
3        28
Name: count, dtype: int64

## Per Phase Analysis

In [7]:
df_benign = df[df["phase"] == 0]
df_phase_1 = df[df["phase"] == 1]
df_phase_2 = df[df["phase"] == 2]
df_phase_3 = df[df["phase"] == 3]
df_phase_4 = df[df["phase"] == 4]   

### Phase 1 - Data Exfiltration

In [8]:
phase_1_labels = df_phase_1["label"].value_counts()
print("Phase 1 Label Distribution:")
print(phase_1_labels)

Phase 1 Label Distribution:
label
data exfiltration    14678
Name: count, dtype: int64


In [12]:
analyze_phase(df_phase_1, "Phase 1")

Total Flows: 14678

 --- IP distribution ---

Source IPs (1):
src_ip
192.168.255.254    14678
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.130.77    14678
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (13109):
sport
6123     4
50163    4
65507    4
35669    4
41516    3
        ..
53576    1
58676    1
23005    1
38452    1
43646    1
Name: count, Length: 13109, dtype: int64

Destination Ports (1):
dport
53    14678
Name: count, dtype: int64


(src_ip
 192.168.255.254    14678
 Name: count, dtype: int64,
 dst_ip
 192.168.130.77    14678
 Name: count, dtype: int64,
 sport
 6123     4
 50163    4
 65507    4
 35669    4
 41516    3
         ..
 53576    1
 58676    1
 23005    1
 38452    1
 43646    1
 Name: count, Length: 13109, dtype: int64,
 dport
 53    14678
 Name: count, dtype: int64)

In [23]:
def analyze_origin_destination(df, phase_name):
    origins = df["local_orig"].value_counts()
    print(f"{phase_name} Origin Distribution:")
    print(origins)

    destinations = df["local_resp"].value_counts()
    print(f"{phase_name} Destination Distribution:")
    print(destinations)

In [24]:
analyze_origin_destination(df_phase_1, "Phase 1")

Phase 1 Origin Distribution:
local_orig
T    14678
Name: count, dtype: int64
Phase 1 Destination Distribution:
local_resp
T    14678
Name: count, dtype: int64


### Phase 2 Analysis - Scanning

In [9]:
print("Number of samples in Phase 2:", len(df_phase_2))

Number of samples in Phase 2: 56443


In [25]:
analyze_origin_destination(df_phase_2, "Phase 2")

Phase 2 Origin Distribution:
local_orig
T    56443
Name: count, dtype: int64
Phase 2 Destination Distribution:
local_resp
T    56434
F        9
Name: count, dtype: int64


In [32]:
analyze_phase(df_phase_2, "Phase 2")

Total Flows: 56443

 --- IP distribution ---

Source IPs (5):
src_ip
172.17.130.196     52306
192.168.130.77      4128
192.168.128.4          5
192.168.128.195        3
192.168.130.110        1
Name: count, dtype: int64

Destination IPs (8204):
dst_ip
10.35.35.206       10666
172.17.130.37       2812
192.168.128.4       2649
10.35.33.111        2488
172.17.129.140      2393
                   ...  
192.168.129.212        2
10.35.35.118           2
192.168.128.170        2
10.35.32.1             1
172.17.128.63          1
Name: count, Length: 8204, dtype: int64

 --- Port distribution ---
Source Ports (13622):
sport
59174    17
59510    16
59944    16
51730    15
34510    15
         ..
41648     1
54762     1
39222     1
54742     1
37206     1
Name: count, Length: 13622, dtype: int64

Destination Ports (1413):
dport
443      24981
80       16526
587         45
993         21
25          20
         ...  
40000        7
53535        7
60123        6
823          6
1212         6
Name: 

(src_ip
 172.17.130.196     52306
 192.168.130.77      4128
 192.168.128.4          5
 192.168.128.195        3
 192.168.130.110        1
 Name: count, dtype: int64,
 dst_ip
 10.35.35.206       10666
 172.17.130.37       2812
 192.168.128.4       2649
 10.35.33.111        2488
 172.17.129.140      2393
                    ...  
 192.168.129.212        2
 10.35.35.118           2
 192.168.128.170        2
 10.35.32.1             1
 172.17.128.63          1
 Name: count, Length: 8204, dtype: int64,
 sport
 59174    17
 59510    16
 59944    16
 51730    15
 34510    15
          ..
 41648     1
 54762     1
 39222     1
 54742     1
 37206     1
 Name: count, Length: 13622, dtype: int64,
 dport
 443      24981
 80       16526
 587         45
 993         21
 25          20
          ...  
 40000        7
 53535        7
 60123        6
 823          6
 1212         6
 Name: count, Length: 1413, dtype: int64)

In [15]:
df_phase_2.columns

Index(['flow_id', 'start_time', 'end_time', 'duration', 'src_ip', 'sport',
       'dst_ip', 'dport', 'proto', 'service', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents', 'ip_proto', 'label', 'sensor_host', 'start_time_dt',
       'end_time_dt', 'phase'],
      dtype='object')

### Phase 3 Analysis - Exploitation

In [26]:
analyze_origin_destination(df_phase_3, "Phase 3")

Phase 3 Origin Distribution:
local_orig
T    28
Name: count, dtype: int64
Phase 3 Destination Distribution:
local_resp
T    28
Name: count, dtype: int64


In [30]:
analyze_phase(df_phase_3, "Phase 3")

Total Flows: 28

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    28
Name: count, dtype: int64

Destination IPs (1):
dst_ip
10.35.35.206    28
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (14):
sport
35148    2
35150    2
35152    2
35154    2
35156    2
35158    2
35160    2
35162    2
35164    2
35166    2
35168    2
35170    2
35172    2
35174    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    28
Name: count, dtype: int64


(src_ip
 172.17.130.196    28
 Name: count, dtype: int64,
 dst_ip
 10.35.35.206    28
 Name: count, dtype: int64,
 sport
 35148    2
 35150    2
 35152    2
 35154    2
 35156    2
 35158    2
 35160    2
 35162    2
 35164    2
 35166    2
 35168    2
 35170    2
 35172    2
 35174    2
 Name: count, dtype: int64,
 dport
 443    28
 Name: count, dtype: int64)

### Phase 4 Analysis - Cracking

In [28]:
analyze_origin_destination(df_phase_4, "Phase 4")

Phase 4 Origin Distribution:
local_orig
T    61
Name: count, dtype: int64
Phase 4 Destination Distribution:
local_resp
T    60
F     1
Name: count, dtype: int64


In [31]:
analyze_phase(df_phase_4, "Phase 4")

Total Flows: 61

 --- IP distribution ---

Source IPs (3):
src_ip
172.17.130.196    56
192.168.128.4      4
10.35.35.206       1
Name: count, dtype: int64

Destination IPs (5):
dst_ip
172.17.130.37      34
10.35.35.206       22
192.168.130.77      3
151.101.14.109      1
192.168.255.254     1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (31):
sport
55754    3
36406    2
35176    2
36408    2
36410    2
35178    2
51940    2
35180    2
36412    2
35182    2
59296    2
36414    2
36416    2
36418    2
36420    2
36422    2
36424    2
36426    2
36428    2
36430    2
36432    2
36434    2
36436    2
50330    2
50328    2
50334    2
50332    2
50336    2
50338    2
51972    1
45119    1
Name: count, dtype: int64

Destination Ports (3):
dport
443    43
80     17
53      1
Name: count, dtype: int64


(src_ip
 172.17.130.196    56
 192.168.128.4      4
 10.35.35.206       1
 Name: count, dtype: int64,
 dst_ip
 172.17.130.37      34
 10.35.35.206       22
 192.168.130.77      3
 151.101.14.109      1
 192.168.255.254     1
 Name: count, dtype: int64,
 sport
 55754    3
 36406    2
 35176    2
 36408    2
 36410    2
 35178    2
 51940    2
 35180    2
 36412    2
 35182    2
 59296    2
 36414    2
 36416    2
 36418    2
 36420    2
 36422    2
 36424    2
 36426    2
 36428    2
 36430    2
 36432    2
 36434    2
 36436    2
 50330    2
 50328    2
 50334    2
 50332    2
 50336    2
 50338    2
 51972    1
 45119    1
 Name: count, dtype: int64,
 dport
 443    43
 80     17
 53      1
 Name: count, dtype: int64)

In [29]:
analyze_origin_destination(df_benign, "Benign")

Benign Origin Distribution:
local_orig
T    400021
F         3
Name: count, dtype: int64
Benign Destination Distribution:
local_resp
T    337453
F     62571
Name: count, dtype: int64


In [27]:
analyze_origin_destination(df, "All Phases")

All Phases Origin Distribution:
local_orig
T    471231
F         3
Name: count, dtype: int64
All Phases Destination Distribution:
local_resp
T    408653
F     62581
Name: count, dtype: int64
